# AG News — EDA

Goal: confirm the real columns, class balance, and text quality before the DATA box gets written.

In [1]:
from pathlib import Path

import pandas as pd

# I keep every path anchored to the project root, never to wherever the kernel happened to start.
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
print(f"columns: {list(train.columns)}")

train.head(3)

train: (120000, 3)
test:  (7600, 3)
columns: ['Class Index', 'Title', 'Description']


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...


In [2]:
print(train["Class Index"].value_counts().sort_index())
print()
print(train.isna().sum())
print()
print(f"exact duplicate rows: {train.duplicated().sum()}")
print()
print(train.loc[2, "Description"])

Class Index
1    30000
2    30000
3    30000
4    30000
Name: count, dtype: int64

Class Index    0
Title          0
Description    0
dtype: int64

exact duplicate rows: 0

Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.


In [3]:
text = train["Title"] + " " + train["Description"]

for pattern in [r"\\", r"&lt;|&gt;|&amp;", r"#\d+;", r"<.*?>"]:
    hits = text.str.contains(pattern, regex=True).sum()
    print(f"{pattern:20} -> {hits:>6} rows  ({hits / len(text):.1%})")

print()
print(text.str.len().describe())

\\                   ->  13146 rows  (11.0%)
&lt;|&gt;|&amp;      ->   5243 rows  (4.4%)
#\d+;                ->  31357 rows  (26.1%)
<.*?>                ->      0 rows  (0.0%)

count    120000.000000
mean        236.460025
std          66.529799
min          17.000000
25%         196.000000
50%         232.000000
75%         266.000000
max        1012.000000
dtype: float64


In [4]:
import html
import re

sample = text.sample(5, random_state=42)

def clean(s: str) -> str:
    s = s.replace("\\", " ")            # the welded escaped newlines
    s = re.sub(r"#(\d+);", r"&#\1;", s) # put back the ampersand something stripped
    s = html.unescape(s)                # &amp; -> &   &#36; -> $
    s = re.sub(r"\s+", " ", s).strip()  # collapse the whitespace I just created
    return s

for raw in sample:
    print("RAW  :", raw[:120])
    print("CLEAN:", clean(raw)[:120])
    print()

RAW  : BBC set for major shake-up, claims newspaper London - The British Broadcasting Corporation, the world #39;s biggest publ
CLEAN: BBC set for major shake-up, claims newspaper London - The British Broadcasting Corporation, the world 's biggest public 

RAW  : Marsh averts cash crunch Embattled insurance broker #39;s banks agree to waive clause that may have prevented access to 
CLEAN: Marsh averts cash crunch Embattled insurance broker 's banks agree to waive clause that may have prevented access to cre

RAW  : Jeter, Yankees Look to Take Control (AP) AP - Derek Jeter turned a season that started with a terrible slump into one of
CLEAN: Jeter, Yankees Look to Take Control (AP) AP - Derek Jeter turned a season that started with a terrible slump into one of

RAW  : Flying the Sun to Safety When the Genesis capsule comes back to Earth with its samples of the sun, helicopter pilots wil
CLEAN: Flying the Sun to Safety When the Genesis capsule comes back to Earth with its samples of the 

In [5]:
LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

for idx, name in LABELS.items():
    print(f"--- {idx} = {name} ---")
    for title in train.loc[train["Class Index"] == idx, "Title"].sample(3, random_state=7):
        print("  ", title)
    print()

--- 1 = World ---
   Kerry touts job-creation plans (AFP)
   Meeting with Manmohan major breakthrough
   Iraq War Foes Focus on Alternative Agenda to Bush (Reuters)

--- 2 = Sports ---
   Smit lauds Boks resolve
   BC, Maine: Chosen two
   AL Wrap: Jays Sink Yankees, Six-Hit Ibanez Mauls Angels (Reuters)

--- 3 = Business ---
   Dollar Rises; Traders Drop Bets Currency to Reach One-Month Low
   US Economy: Growth Slowed Less Than Expected in 2nd Quarter
   Billionaire club grows, Forbes finds

--- 4 = Sci/Tech ---
   Separate Genes Responsible for Drinking, Alcoholism
   Avici extends AT T contract
   Walking May Ward Off Alzheimer's Disease



In [6]:
import sys
from pathlib import Path

# The notebook lives in notebooks/, so the project root isn't on the path by default.
# I add it once so the notebook imports the exact same code the API and tests do.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [7]:
from newsvane.data import get_articles

rows = get_articles("test")
print(len(rows))
print(rows[0])

7600
{'text': "Fears for T N pension after talks Unions representing workers at Turner Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'topic': 'Business', 'timestamp': datetime.datetime(2026, 7, 13, 6, 10, 55, 768741, tzinfo=datetime.timezone.utc)}
